# 02 - Nettoyage et Unification des Données

## Objectif
Ce notebook transforme les trois datasets bruts en un dataset unifié avec un schéma commun.

### Étapes :
1. Nettoyage de chaque dataset individuellement
2. Mapping vers un schéma commun
3. Création du dataset unifié
4. Validation et sauvegarde

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from src.data_processing import (
    load_titanic, load_lusitania, load_sewol,
    clean_titanic, clean_lusitania, clean_sewol,
    create_unified_dataset, save_processed_data
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)

print('Modules chargés.')

---
## 1. Chargement et Nettoyage du Titanic

In [ ]:
titanic_raw = load_titanic()
titanic_clean = clean_titanic(titanic_raw)

print(f"Titanic brut : {titanic_raw.shape}")
print(f"Titanic nettoyé : {titanic_clean.shape}")
print(f"\nColonnes : {titanic_clean.columns.tolist()}")
print(f"\nAperçu :")
titanic_clean.head()

In [ ]:
print("=== Vérification du nettoyage Titanic ===")
print(f"Survived  : {titanic_clean['survived'].value_counts().to_dict()}")
print(f"Sex       : {titanic_clean['sex'].value_counts().to_dict()}")
print(f"Classe    : {titanic_clean['social_class'].value_counts().to_dict()}")
print(f"Is_crew   : {titanic_clean['is_crew'].value_counts().to_dict()}")
print(f"Age       : min={titanic_clean['age'].min():.1f}, max={titanic_clean['age'].max():.1f}, "
      f"NaN={titanic_clean['age'].isna().sum()}")
print(f"Group_size: min={titanic_clean['group_size'].min()}, max={titanic_clean['group_size'].max()}, "
      f"mean={titanic_clean['group_size'].mean():.2f}")

---
## 2. Chargement et Nettoyage du Lusitania

In [ ]:
lusitania_raw = load_lusitania()
lusitania_clean = clean_lusitania(lusitania_raw)

print(f"Lusitania brut : {lusitania_raw.shape}")
print(f"Lusitania nettoyé : {lusitania_clean.shape}")
print(f"\nColonnes : {lusitania_clean.columns.tolist()}")
print(f"\nAperçu :")
lusitania_clean.head()

In [ ]:
print("=== Vérification du nettoyage Lusitania ===")
print(f"Survived  : {lusitania_clean['survived'].value_counts().to_dict()}")
print(f"Sex       : {lusitania_clean['sex'].value_counts().to_dict()}")
print(f"Classe    : {lusitania_clean['social_class'].value_counts().to_dict()}")
print(f"Is_crew   : {lusitania_clean['is_crew'].value_counts().to_dict()}")
print(f"Age       : min={lusitania_clean['age'].min():.1f}, max={lusitania_clean['age'].max():.1f}, "
      f"NaN={lusitania_clean['age'].isna().sum()}")
print(f"Group_size: min={lusitania_clean['group_size'].min()}, max={lusitania_clean['group_size'].max()}, "
      f"mean={lusitania_clean['group_size'].mean():.2f}")

---
## 3. Chargement et Nettoyage du Sewol

In [ ]:
sewol_raw = load_sewol()
sewol_clean = clean_sewol(sewol_raw)

print(f"Sewol brut : {sewol_raw.shape}")
print(f"Sewol nettoyé : {sewol_clean.shape}")
print(f"\nColonnes : {sewol_clean.columns.tolist()}")
print(f"\nAperçu :")
sewol_clean.head()

In [ ]:
print("=== Vérification du nettoyage Sewol ===")
print(f"Survived  : {sewol_clean['survived'].value_counts().to_dict()}")
print(f"Sex       : {sewol_clean['sex'].value_counts().to_dict()}")
print(f"Classe    : {sewol_clean['social_class'].value_counts().to_dict()}")
print(f"Is_crew   : {sewol_clean['is_crew'].value_counts().to_dict()}")
print(f"Age       : min={sewol_clean['age'].min():.1f}, max={sewol_clean['age'].max():.1f}, "
      f"NaN={sewol_clean['age'].isna().sum()}")
print(f"Group_size: min={sewol_clean['group_size'].min()}, max={sewol_clean['group_size'].max()}, "
      f"mean={sewol_clean['group_size'].mean():.2f}")

---
## 4. Création du Dataset Unifié

In [ ]:
unified = create_unified_dataset(titanic_clean, lusitania_clean, sewol_clean)

print(f"Dataset unifié : {unified.shape}")
print(f"\nColonnes : {unified.columns.tolist()}")
print(f"\nDistribution par catastrophe :")
print(unified['disaster'].value_counts())
print(f"\nTaux de survie par catastrophe :")
print(unified.groupby('disaster')['survived'].mean().round(4))
unified.head()

In [ ]:
# Vérification de la qualité du dataset unifié
print("=== Qualité du dataset unifié ===")
print(f"\nValeurs manquantes :")
missing = unified.isnull().sum()
for col in missing[missing > 0].index:
    print(f"  {col}: {missing[col]} ({missing[col]/len(unified)*100:.1f}%)")

print(f"\nStatistiques descriptives :")
unified.describe()

In [ ]:
# Visualisation du dataset unifié
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Dataset Unifié Multi-Catastrophes', fontsize=16, fontweight='bold')

# Taille par catastrophe
unified['disaster'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['steelblue', 'coral', 'mediumseagreen'])
axes[0, 0].set_title('Échantillons par Catastrophe')
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=0)

# Survie par catastrophe
unified.groupby('disaster')['survived'].mean().plot(kind='bar', ax=axes[0, 1], color=['steelblue', 'coral', 'mediumseagreen'])
axes[0, 1].set_title('Taux de Survie par Catastrophe')
axes[0, 1].set_ylabel('Taux')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)

# Distribution d'âge par catastrophe
for disaster in ['titanic', 'lusitania', 'sewol']:
    subset = unified[unified['disaster'] == disaster]['age'].dropna()
    subset.hist(bins=25, ax=axes[0, 2], alpha=0.5, label=disaster.capitalize())
axes[0, 2].set_title('Distribution d\'Âge par Catastrophe')
axes[0, 2].legend()

# Classe sociale par catastrophe
pd.crosstab(unified['disaster'], unified['social_class'], normalize='index').plot(
    kind='bar', ax=axes[1, 0], stacked=True)
axes[1, 0].set_title('Répartition Classe Sociale')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)
axes[1, 0].legend(title='Classe', labels=['Haute', 'Moyenne', 'Basse'])

# Sexe par catastrophe
pd.crosstab(unified['disaster'], unified['sex'], normalize='index').plot(
    kind='bar', ax=axes[1, 1], stacked=True, color=['#3498db', '#e91e8c'])
axes[1, 1].set_title('Répartition par Sexe')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)

# Taille de groupe par catastrophe
for disaster in ['titanic', 'lusitania', 'sewol']:
    subset = unified[unified['disaster'] == disaster]['group_size']
    axes[1, 2].hist(subset, bins=range(1, 12), alpha=0.5, label=disaster.capitalize())
axes[1, 2].set_title('Distribution Taille de Groupe')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

---
## 5. Sauvegarde des Données Nettoyées

In [ ]:
# Sauvegarder chaque dataset nettoyé individuellement
save_processed_data(titanic_clean, 'titanic_cleaned.csv')
save_processed_data(lusitania_clean, 'lusitania_cleaned.csv')
save_processed_data(sewol_clean, 'sewol_cleaned.csv')

# Sauvegarder le dataset unifié
save_processed_data(unified, 'unified_dataset.csv')

print("\nToutes les données ont été sauvegardées dans data/processed/")

---
## 6. Schéma Unifié - Documentation

### Colonnes du dataset unifié

| Colonne | Type | Description | Source Titanic | Source Lusitania | Source Sewol |
|---------|------|-------------|----------------|------------------|-------------|
| `survived` | int (0/1) | Variable cible | Survived | Fate → 0/1 | Raw → 0/1 |
| `age` | float | Âge en années | Age | Age (nettoyé des '?') | age |
| `sex` | str | Genre | Sex | Sex | gender |
| `social_class` | int (1-3) | Classe sociale unifiée | Pclass | Dept/Class → mapping | Category-1 → mapping |
| `is_crew` | int (0/1) | Membre d'équipage | 0 (tous passagers) | Passenger/Crew | Category-1=='sailor' |
| `group_size` | int | Taille du groupe | SibSp+Parch+1 | Traveling Companions | Category-3 → estimation |
| `disaster` | str | Identifiant catastrophe | 'titanic' | 'lusitania' | 'sewol' |
| `disaster_year` | int | Année | 1912 | 1915 | 2014 |
| `is_alone` | int (0/1) | Voyage seul | group_size==1 | group_size==1 | group_size==1 |
| `is_child` | int (0/1) | Moins de 18 ans | age<18 | age<18 | age<18 |
| `sex_binary` | int (0/1) | Sexe encodé (1=F) | sex=='female' | sex=='female' | sex=='female' |

### Décisions de mapping

**Classe sociale** :
- Titanic : Pclass directement (1=Saloon, 2=Second, 3=Third)
- Lusitania : Saloon→1, Second→2, Third→3, Crew→2 (classe moyenne par défaut)
- Sewol : Équipage→2, Étudiants→3 (pas de classe haute dans un ferry), Passagers normaux→2

**Taille de groupe** :
- Titanic : SibSp + Parch + 1 (informations familiales précises)
- Lusitania : Estimation depuis les notes de voyage (approximation)
- Sewol : Estimation depuis Category-3 (family→3, group→4, half→30, personal→1)